In [ ]:
import streamlit as st
import pandas as pd
import joblib
import numpy as np

# --- 1. Load the ML Brain ---
# Make sure your .joblib files are in the same folder as this script!
@st.cache_resource
def load_assets():
    model = joblib.load('risk_model.joblib')
    scaler = joblib.load('scaler.joblib')
    le = joblib.load('label_encoder.joblib')
    return model, scaler, le

# --- 2. Frontend Layout ---
st.set_page_config(page_title="Mentor Connect", page_icon="🎓")
st.title("🎓 Mentor-Mentee ML Portal")

model, scaler, le = load_assets()

# --- 3. Sidebar Navigation ---
menu = st.sidebar.selectbox("Menu", ["Single Student Check", "Batch Processing"])

if menu == "Single Student Check":
    st.subheader("Enter Student Details")
    
    with st.form("prediction_form"):
        col1, col2 = st.columns(2)
        with col1:
            attn = st.slider("Attendance %", 0, 100, 75)
            cgpa = st.number_input("CGPA", 0.0, 10.0, 7.0)
            backlogs = st.number_input("Backlogs", 0, 5, 0)
            dc = st.number_input("Disciplinary Cases", 0, 5, 0)
        with col2:
            leaves = st.number_input("Leaves Taken", 0, 20, 2)
            late = st.number_input("Late Night Entries", 0, 20, 1)
            missed = st.number_input("Missed Consultations", 0, 10, 0)
            stress = st.slider("Stress Level (1-10)", 1, 10, 5)
            sentiment = st.slider("Sentiment Score", 0.0, 1.0, 0.5)
        
        submit = st.form_submit_button("Predict Risk Level")

    if submit:
        # Prepare data for model
        features = np.array([[attn, cgpa, backlogs, dc, leaves, late, missed, stress, sentiment]])
        features_scaled = scaler.transform(features)
        
        # Predict
        pred = model.predict(features_scaled)
        risk = le.inverse_transform(pred)[0]
        
        # UI Response
        if risk == 'High':
            st.error(f"Prediction: {risk} Risk. Schedule a meeting immediately.")
        elif risk == 'Medium':
            st.warning(f"Prediction: {risk} Risk. Monitor closely.")
        else:
            st.success(f"Prediction: {risk} Risk. Student is doing well!")

elif menu == "Batch Processing":
    st.subheader("Upload Student Dataset")
    # FIX: Corrected attribute name below
    uploaded_file = st.file_uploader("Upload your CSV file", type=["csv"])
    
    if uploaded_file is not None:
        data = pd.read_csv(uploaded_file)
        st.write("Data Preview:", data.head())
        
        if st.button("Generate Prioritized Action Plan"):
            # Backend processing logic
            X = data.drop(['roll_no', 'risk_level'], axis=1, errors='ignore')
            X_scaled = scaler.transform(X)
            data['Predicted_Risk'] = le.inverse_transform(model.predict(X_scaled))
            
            st.write("Processed Results:")
            st.dataframe(data.sort_values(by='Predicted_Risk'))
            
            # Export button
            csv = data.to_csv(index=False).encode('utf-8')
            st.download_button("Download Action Plan", csv, "tutor_plan.csv", "text/csv")

In [2]:
import os
print(os.getcwd())

C:\Users\KIIT0001\Tutormentor
